# 2026-2027 Python для дата-аналітики: від обробки до візуалізації

## Практична робота №2: Робота з pandas на даних про акції FAANG

### **ТЕМА:**
Робота з бібліотекою pandas: об'єднання наборів даних, конвертація типів, сортування, фільтрація, пошук екстремальних значень та переформатизація
даних на прикладі набору даних про акції компаній Facebook, Apple, Amazon, Netflix та Google (FAANG).

### **МЕТА:**
Мета практичної роботи полягає у відпрацюванні навичок роботи з бібліотекою pandas для обробки кількох табличних наборів даних. Студенти навчаться
об'єднувати файли, проводити конвертацію типів даних, сортувати та фільтрувати дані, знаходити екстремальні значення, а також перетворювати дані з
wide-формату в long-формат за допомогою методу melt() . У роботі використовується набір даних про акції FAANG (файли aapl.csv , amzn.csv ,
fb.csv , goog.csv , nflx.csv ), що дозволяє застосувати теоретичні знання на реальних фінансових даних.

### **ЗАВДАННЯ:**
Використовуючи надані файли з даними про акції ( aapl.csv , amzn.csv , fb.csv , goog.csv , nflx.csv ) та матеріали з лекційного розділу про pandas,
виконайте наступні вправи. Робота проводиться в середовищі Jupyter Notebook (або аналогічному IDE з підтримкою Python). Перед початком завантажте
дані за допомогою команди pd.read_csv() для кожного файлу.
Ознайомтеся зі структурою даних: колонками (наприклад, date , open , high , low , close , volume ) та типами даних. Виконайте необхідне очищення
даних (наприклад, обробку пропусків, якщо вони є).



**ПІБ:** Потапова Інна Станіславівна\
**Група:** м2ЦТ-з\
**Дата:** 17.11.2026
---

In [119]:
# Визначення версії пайтону, потрібна 3.12+
!python -V
# Встановлення необхідних pandas
!pip install pandas


Python 3.12.16



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [120]:
#### Опис данних
import os
from enum import Enum

import pathlib
import pandas as pd

DIR = os.getcwd()

class Faang(Enum):
    """Enum для компаній FAANG з шляхами до CSV файлів."""

    AAPL = pathlib.Path(DIR) / "aapl.csv"
    AMZN = pathlib.Path(DIR) / "amzn.csv"
    FB = pathlib.Path(DIR) / "fb.csv"
    GOOG = pathlib.Path(DIR) / "goog.csv"
    NFLX = pathlib.Path(DIR) / "nflx.csv"

def read_data(company: Faang) -> pd.DataFrame:
    """Зчитує дані з CSV файлу для заданої компанії FAANG та виводить основну інформацію про таблицю.
    
    :param company: Об'єкт Enum Faang, що вказує на компанію, дані якої потрібно зчитати.
    :return: DataFrame з даними компанії.
    """
    data = pd.read_csv(company.value)
    print(f"Зразок таблиці для компанії {company.name}:", data.head(), sep="\n")
    print("\n", f"Статистичний опис таблиці для компанії {company.name}:", data.describe(), sep="\n")
    print("\n", f"Імена колонок для компанії {company.name}:", data.columns, sep="\n")
    print("\n", f"Розмір таблиці для компанії {company.name}:", data.shape, sep="\n")
    return data


company_data = {}
for company in Faang:
    if not company.value.exists():
        raise FileNotFoundError(f"CSV файл не знайдено для компанії {company.name} по шляху: {company.value}")
    company_data[company.name] = read_data(company=company)


Зразок таблиці для компанії AAPL:
         date       high        low       open      close       volume
0  2018-01-02  43.075001  42.314999  42.540001  43.064999  102223600.0
1  2018-01-03  43.637501  42.990002  43.132500  43.057499  118071600.0
2  2018-01-04  43.367500  43.020000  43.134998  43.257500   89738400.0
3  2018-01-05  43.842499  43.262501  43.360001  43.750000   94640000.0
4  2018-01-08  43.902500  43.482498  43.587502  43.587502   82271200.0


Статистичний опис таблиці для компанії AAPL:
             high         low        open       close        volume
count  251.000000  251.000000  251.000000  251.000000  2.510000e+02
mean    47.748526   46.795877   47.277859   47.263357  1.360803e+08
std      5.146255    5.109996    5.126908    5.148465  5.850376e+07
min     37.887501   36.647499   37.037498   36.707500  5.005560e+07
25%     43.737499   43.005001   43.356251   43.285000  9.418680e+07
50%     46.849998   46.235001   46.572498   46.512501  1.266524e+08
75%     52.343750

#### Вправа 1

Об'єднайте дані про акції FAANG у єдиний набір даних та збережіть його у файл faang.csv . Збережіть DataFrame як faang для подальших вправ: a)
Завантажте файли aapl.csv , amzn.csv , fb.csv , goog.csv та nflx.csv за допомогою pd.read_csv() . b) Додайте до кожного DataFrame стовпець
ticker , що вказує на тікер акції (наприклад, 'AAPL' для Apple; імена файлів відповідають тікерам). c) Об'єднайте всі DataFrame у єдиний за допомогою
pd.concat() або pd.append() (зверніть увагу на параметр ignore_index ). d) Збережіть результат у файл faang.csv за допомогою to_csv().

Виведіть перші 5 рядків faang для перевірки.

In [121]:
# проход циклом по компаніях FAANG та з'єднання їх даних у єдиний датафрейм
faang = pd.DataFrame()
for company in Faang:
    ticker = company.name
    company_data[ticker]["ticker"] = ticker
    faang = pd.concat([faang, company_data[ticker]], ignore_index=True)

# зберегти об'єднаний датафрейм у CSV файл
faang.to_csv("faang.csv")

# перегляд перших 5 рядків об'єднаного датафрейму
faang.head()


,date,high,low,open,close,volume,ticker
0,2018-01-02,43.075001,42.314999,42.540001,43.064999,102223600.0,AAPL
1,2018-01-03,43.637501,42.990002,43.132500,43.057499,118071600.0,AAPL
2,2018-01-04,43.367500,43.020000,43.134998,43.257500,89738400.0,AAPL
3,2018-01-05,43.842499,43.262501,43.360001,43.750000,94640000.0,AAPL
4,2018-01-08,43.902500,43.482498,43.587502,43.587502,82271200.0,AAPL


### Вправа 2

За допомогою faang проведіть конвертацію типів даних: перетворіть колонку date на datetime за допомогою pd.to_datetime() , а колонку volume на
цілі числа за допомогою astype(int) . Потім відсортуйте дані за колонками date та ticker (використовуйте sort_values() ). Виведіть перші 10
рядків відсортованого DataFrame.


In [122]:
#  перетворення дати у datetime та volume у int
faang["date"] = pd.to_datetime(faang["date"])
faang["volume"] = faang["volume"].astype(int)

# сортування за date та ticker
faang = faang.sort_values(by=["date", "ticker"])

# перегляд перших 10 рядків відсортованого датафрейму
faang.head(10)


,date,high,low,open,close,volume,ticker
0,2018-01-02,43.075001,42.314999,42.540001,43.064999,102223600,AAPL
251,2018-01-02,1190.000000,1170.510010,1172.000000,1189.010010,2694500,AMZN
502,2018-01-02,181.580002,177.550003,177.679993,181.419998,18151900,FB
753,2018-01-02,1066.939941,1045.229980,1048.339966,1065.000000,1237600,GOOG
1004,2018-01-02,201.649994,195.419998,196.100006,201.070007,10966900,NFLX
1,2018-01-03,43.637501,42.990002,43.132500,43.057499,118071600,AAPL
252,2018-01-03,1205.489990,1188.300049,1188.300049,1204.199951,3108800,AMZN
503,2018-01-03,184.779999,181.330002,181.880005,184.669998,16886600,FB
754,2018-01-03,1086.290039,1063.209961,1064.310059,1082.479980,1430200,GOOG
1005,2018-01-03,206.210007,201.500000,202.050003,205.050003,8591400,NFLX


#### Вправа 3

Знайдіть 7 рядків у faang з найнижчими значеннями у колонці volume . Використовуйте сортування за volume (метод sort_values() ) та вибір перших
7 рядків ( head(7) ). Виведіть ці рядки з коментарем про можливі причини низького обсягу торгівлі.


In [123]:
faang.nsmallest(7, "volume").head(7)

,date,high,low,open,close,volume,ticker
879,2018-07-03,1135.819946,1100.020020,1135.819946,1102.890015,679000,GOOG
979,2018-11-23,1037.589966,1022.398987,1030.000000,1023.880005,691500,GOOG
852,2018-05-24,1080.469971,1066.150024,1079.000000,1079.239990,766800,GOOG
883,2018-07-10,1159.589966,1149.589966,1156.979980,1152.839966,798400,GOOG
905,2018-08-09,1255.541992,1246.010010,1249.900024,1249.099976,848600,GOOG
912,2018-08-20,1211.000000,1194.625977,1205.020020,1207.770020,870800,GOOG
914,2018-08-22,1211.839966,1199.000000,1200.000000,1207.329956,887400,GOOG


Аналіз причин найменшого volume:

1️⃣ День тижня - Святкові дні та вихідні

2018-07-03 (Вівторок) - 1 день перед День Незалежності США (4 липня) → людьми часто беруть довгий вікенд\
2018-11-23 (Пятниця) - День подяки (Thanksgiving) у США → виходячи на вікенд і святкування\
2018-05-24 (Четвер) - День перед Memorial Day (понеділок, 28 травня) → довгий вікенд\
2018-07-10 (Вівторок) - Після вікенду в період літнього затишку\
2018-08-09 (Четвер) - Четвер у період літнього відпусток\
2018-08-20 (Понеділок) - Понеділок у період літнього затишку\
2018-08-22 (Середа) - Середа в період літнього затишку

2️⃣ Сезонність та волатильність

Липень-Серпень (літо) - Період "summer doldrums" на фондовому ринку, коли багато інвесторів на відпустці\
Менше трейдерів активно торгує\
Нижчий інтерес до покупок/продажів

#### Вправа 4

Припустіть, що 26 липня 2018 року сталася помилка в записі даних (glitch). Обговоріть, як це впливає на аналіз (наприклад, на статистики чи
візуалізацію). Запропонуйте стратегії обробки: видалення рядків, заміна значень чи позначення як outliers. Не пишіть код — надайте текстове пояснення з
прикладами можливих наслідків. Виведіть рядок(и) з датою 26 липня 2018 для ілюстрації.

In [124]:
glitch = faang[faang['date'] == '2018-07-26']
glitch.head(10)

,date,high,low,open,close,volume,ticker
142,2018-07-26,48.990002,48.402500,48.652500,48.552502,76304000,AAPL
393,2018-07-26,1844.680054,1804.500000,1839.000000,1808.000000,9924400,AMZN
644,2018-07-26,180.130005,173.750000,174.889999,176.259995,169803700,FB
895,2018-07-26,1269.770996,1249.020020,1251.000000,1268.329956,2405600,GOOG
1146,2018-07-26,365.540009,356.630005,358.190002,363.089996,6993700,NFLX


У випадку якщо ми вважаємо що данні за 2018-07-26 вибиваються з загальної картини (outliers), під час виклику методу зведення статистик середнє арифметичне зміщується в бік помилкового значення, а мінімальне чи максимальне значення повністю втрачає зв'язок із реальністю. Наприклад, якщо ціна акції Amazon випадково записалася як нуль (або NaN з конвертацією), середня ціна за рік штучно занизиться, а реальні історичні мінімуми стануть непомітними.

Стрибок стандартного відхилення (Standard Deviation). Оскільки стандартне відхилення обчислюється через квадрат різниці між кожним значенням і середнім, один екстремальний викид багаторазово збільшує цей показник. У фінансовому аналізі це створює хибну ілюзію надвисокої волатильності та ризикованості активу.

Бажано було б створити matplotlib графік який би наочно показав девіацію у данних саме у цей день. Корисним був би графік розподілу залишків та Q-Q Plot (квантиль-квантиль). 

Способи виправлення:

1. Повне відфільтровування дати 26 липня 2018 року з індексу таблиц. У цьому випадку порушується регулярність часового індексу, але аутлайери не впливають і не створюють аномальну волотільність

2. Заповнення збою попереднім значенням (Forward Fill), медіаною або лінійною інтерполяцією між сусідніми днями. Добре підходить для подальшого вирішення задач з лінійної регресії

3. Створення додаткового булевого стовпця-прапорця або переведення помилкових значень у статус пропущених (NaN) без видалення самої дати.

### Контрольні питання:

**1.Яка основна відмінність між методами pd.concat() та pd.merge() при об'єднанні DataFrame? Коли використовувати кожен?**

**Відповідь:**

pd.concat() просто додає рядки та стовпці один за іншим, доволі швидка операція, не порівнює данні. Схожий з UNION SQL операцією, може мати дубльовані індекси

pd.merge() працюэ як sql join та знаходить спільні ключи для з'єднення данних з різних джерел, може працювати по inner, outer, left, right, новий індекс залежить від результату

**2. Як конвертувати колонку з рядками у datetime в pandas? Наведіть приклад коду з обробкою помилок (параметр errors ).**

**Відповідь:**


```python
import pandas as pd

df = pd.DataFrame({
    'date': ['2023-01-15', 'invalid-date', '2023-03-10', None]
})

# Конвертує помилки в NaT
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df)
#         date
# 0 2023-01-15
# 1       NaT  ← помилка замінена на NaT
# 2 2023-03-10
# 3       NaT  ← None також NaT

df = pd.DataFrame({
    'date': ['2023-01-15', 'invalid-date', '2023-03-10']
})

# Перехоплення помилки
df = pd.DataFrame({
    'date': ['2023-01-15', 'invalid-date', '2023-03-10']
})

try:
    df['date'] = pd.to_datetime(df['date'], errors='raise')
except Exception as e:
    print(f"Помилка: {e}")
    # Помилка: Unknown string format: invalid-date
```

**3. Поясніть, що таке wide- та long-формати даних.**

**Відповідь:**

##### WIDE FORMAT (широкий)

Кожна змінна — окремий стовпець
Більше стовпців, менше рядків
Зручна для читання та аналізу людиною

| Student | Math | English | Science |
|---------|------|---------|---------|
| Alice   | 95   | 88      | 92      |
| Bob     | 87   | 95      | 89      |
| Carol   | 92   | 91      | 95      |


##### LONG FORMAT (довгий)

Кожне значення — окремий рядок
Менше стовпців, більше рядків
Зручна для обробки та анлітики (ggplot2, статистика)

| Student | Subject | Score |
|---------|---------|-------|
| Alice   | Math    | 95    |
| Alice   | English | 88    |
| Alice   | Science | 92    |
| Bob     | Math    | 87    |
| Bob     | English | 95    |
| Bob     | Science | 89    |
| Carol   | Math    | 92    |
| Carol   | English | 91    |
| Carol   | Science | 95    |


```

**4. Як знайти рядки з найменшими значеннями у колонці без повного сортування (використовуйте nsmallest() )? Наведіть приклад**

**Відповідь:**

```python
import pandas as pd

df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David'],
    'salary': [5000, 3000, 7000, 2500]
})

# Знайти 2 найменші зарплати
print(df.nsmallest(2, 'salary'))
#    name  salary
# 3 David    2500
# 1   Bob    3000
```

**5. Чому важливо сортувати дані за датою перед аналізом часових рядів?.**

**Відповідь:**

Послідовність має значення — часові ряди залежать від порядку точок даних. Перемішані дати дають невірні результати.

Функції rolling/cumsum вимагають порядку — вибірка середнього, накопичена сума, диференціювання працюють тільки в хронологічному порядку.

Тренди та сезонності видно тільки в порядку — без сортування графіки будуть хаосом.

Прогнозування неможливе — моделі ARIMA, експоненціальне згладжування вимагають хронологічного порядку.

**6. Як обробити можливі помилки в даних (наприклад, glitch), такі як аномальні значення? Які етичні аспекти видалення даних можуть бути?**

**Відповідь:**

#### Методи виявлення аномалій

#### 1. IQR (Міжквартильний розмах)
Визначає значення, що виходять за межі Q1 - 1.5×IQR та Q3 + 1.5×IQR. Простий і надійний для нормально розподілених даних.

#### 2. Z-score (Стандартне відхилення)
Значення з абсолютним Z-score > 3 вважаються аномаліями. Вимагає нормального розподілу.

#### 3. Isolation Forest (Машинне навчання)
Алгоритм, що ізолює аномалії на основі випадкових поділів. Не вимагає припущень про розподіл.

#### Способи обробки аномалій

| Метод | Коли використовувати |
|-------|------------------|
| **Видалити** | Явна помилка, дані непотрібні |
| **Замінити на медіану** | Мало аномалій, типова помилка введення |
| **Замінити на NaN** | Невідомо чи коректне значення |
| **Залишити** | Це реальні екстремальні значення |

#### ⚠️ Етичні аспекти видалення даних

#### Упередженість (Bias)
Видалення аномалій може скасувати справжні групи людей або подій. Наприклад, видалення "аномально" низьких доходів може приховати реальну бідність та дискримінацію. 

#### Репрезентативність
Видалення змінює розподіл даних. Результати можуть не узагальнюватись на цілу популяцію данних. Важливо документувати скільки відсотків даних видалено.

#### Прозорість
Видалення даних має бути явно описано у документації. Видалення данних без документування може створити проблеми для інших дослідників, які працюють з виправленими даними.

**7. Яке значення має колонка volume (тип даних, одиниці виміру), і як конвертація на int впливає на обчислення статистики?**

**Відповідь:**

Volume — це кількість акцій, що були куплені/продані в певний період (день, годину).
При першому завантажені volume має формат np.float64, після перетворення np.int64. Дозволило округлити данні та мінімізувати округлення помилок, дало економія місця в пам'яті.

Але, як на мене, замість завчасного преведення типу у інт як вимагається вправами, що призводить до округлення, ми могли б перевести все у формат float.64, щоб не втрачати точність, не акамулювати помилки, не зміщувати mean/median, не зменшувати дисперсію та не допустити недооцінення варіативності. А у інт переводити лише в кінці на стадії аналізу данних та очистки від NaN, адже, на відміну від float64, int64 не працює з NaN (альтернатива Int64)